# S2 cointegration — FTMO 2-Step challenge select

Compare **FTMO Challenge 2-Step** (not 1-step) pass probability and **economic EV** (fee, payout split, retries). This is **not** $P(\mathrm{beat\ SPY})$ — that lives under `06_risk/monte_carlo/`.

Fees, account size, profit split, and horizon $H$ are **inputs** (not official FTMO prices). Official 2-step rules encoded here have no calendar time cap; $H$ is the simulation window. `incomplete` at $H$ is not a rule violation.


## 0. Imports & Config


In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.monte_carlo.loaders import find_repo_root, load_sealed_s1, load_sealed_s2
from risk.prop_firm.registry import CHALLENGES, make_challenge
from risk.prop_firm.report import binding_mix, leverage_ev_grid, run_challenge_select
from risk.prop_firm.s1_calendar import weekly_to_weekday_returns
from risk.prop_firm.plots import failure_mix_figure, leverage_heatmap_figure, retries_hist_figure

ROOT = find_repo_root(ROOT)
print("ROOT", ROOT)
print("registered challenges", sorted(CHALLENGES))

SLEEVE = 's2'
FIRM_KEY = 'ftmo.2step'


## 1. Data Loading


In [ ]:
RETURNS = load_sealed_s2(ROOT)
DEFAULT_H = 40
DEFAULT_HF = 40
print("daily bars", len(RETURNS), RETURNS.index.min().date(), RETURNS.index.max().date())
print(RETURNS.tail())


## 2. Challenge selector

v1 registry: `ftmo.2step.challenge` / `.verification` / `.funded`. Later firms register new classes without rewriting this notebook.

S2 sealed series is already **daily**. EOD book returns still cannot see true CEST intra-day equity or the calendar day a pair was opened; `|r|>eps` is the proxy.


In [ ]:
print(make_challenge("ftmo.2step.challenge").name())
print("challenge target", make_challenge("ftmo.2step.challenge").profit_target_frac())
print("verification target", make_challenge("ftmo.2step.verification").profit_target_frac())
print("funded target", make_challenge("ftmo.2step.funded").profit_target_frac())


## 3. Pass rates & economic EV


In [ ]:
PACK = {"headline": None, "results": None}

def run(n_simulations, horizon, horizon_funded, leverage, initial_capital, fee, profit_split, mean_block_length):
    pack = run_challenge_select(
        RETURNS,
        n_simulations=int(n_simulations),
        horizon=int(horizon),
        horizon_funded=int(horizon_funded),
        leverage=float(leverage),
        initial_capital=float(initial_capital),
        fee=float(fee),
        profit_split=float(profit_split),
        mean_block_length=float(mean_block_length),
        random_seed=0,
    )
    PACK.update(pack)
    display(pack["headline"].to_frame("value"))
    print("do_not_take (EV<=0 or lower CI<=0):", bool(pack["headline"]["do_not_take"]))
    pack["results"].head()
    return pack

try:
    import ipywidgets as w
    ui = w.interactive(
        run,
        n_simulations=w.IntSlider(min=50, max=1500, value=300, step=50, description="n_sim"),
        horizon=w.IntSlider(min=10, max=180, value=DEFAULT_H, step=5, description="H eval"),
        horizon_funded=w.IntSlider(min=10, max=180, value=DEFAULT_HF, step=5, description="H funded"),
        leverage=w.FloatSlider(min=0.25, max=3.0, value=1.0, step=0.25, description="k"),
        initial_capital=w.Dropdown(options=[10000.0, 25000.0, 50000.0, 100000.0, 200000.0], value=100000.0, description="size"),
        fee=w.FloatSlider(min=0.0, max=2000.0, value=540.0, step=20.0, description="fee"),
        profit_split=w.FloatSlider(min=0.5, max=0.9, value=0.8, step=0.05, description="split"),
        mean_block_length=w.FloatSlider(min=2.0, max=25.0, value=10.0, step=1.0, description="block L"),
    )
    display(ui)
except Exception as exc:
    print("ipywidgets unavailable (%s); running defaults" % exc)
    run(300, DEFAULT_H, DEFAULT_HF, 1.0, 100000.0, 540.0, 0.8, 10.0)


## 4. Evaluation

$P(\mathrm{challenge})$, $P(\mathrm{verification})$, $P(\mathrm{both})$, median/p90 days-to-pass, failure mix, economic EV with percentile CI and $P(\mathrm{EV}\le 0)$, EV/day, do-not-take. Optimize **EV/day**, not raw pass rate.


In [ ]:
if PACK.get("results") is not None:
    mix = binding_mix(PACK["results"], phase="chal")
    display(mix.to_frame("n"))
    failure_mix_figure(mix).show()
    retries_hist_figure(PACK["retries_until_pass"]).show()
else:
    print("Run the selector in section 3 first.")
